# Hurtownia Danych Iowa Liquor Sales: Kompleksowy przewodnik po Architekturze i Transformacjach
Ten notatnik to dokumentacja obronna projektu badająca dogłębnie architekturę ETL oraz analityczną. 
Projekt opiera się na **Architekturze Medalionowej (Medallion Architecture)** i założeniach analityki **OLAP (Online Analytical Processing)**. 
Będziemy analizować 3 warstwy:
1. **Warstwa Brązowa (Bronze / Staging)** - Zderzenie z surową, niefiltrowaną rzeczywistością danych źródłowych i omówienie wyzwań integracyjnych.
2. **Warstwa Srebrna (Silver / Data Warehouse)** - Wielowymiarowy Model Gwiazdy. Zaprezentujemy techniki deduplikacji, kluczy zastępczych, łatania braków danych (imputacji) oraz funkcje analityczne SQL.
3. **Warstwa Złota (Gold / Semantic Layer)** - Składa się na nią aż **16 zoptymalizowanych widoków SQL** pełniących rolę dedykowanych martów danych dla różnorodnych domen raportowych (Czas, Geografia, Asortyment, Wskaźniki biznesowe).


In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import pandas as pd
from src.utils.db import sqlserver_connection
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
def query_db(sql_query: str) -> pd.DataFrame:
    with sqlserver_connection() as conn:
        return pd.read_sql(sql_query, conn)


## 1. Warstwa Brązowa (Bronze / Staging) - Jak wyglądają surowe dane?
Pierwszym krokiem każdego procesu ETL jest Ekstrakcja (Extract) i Ładowanie (Load) do warstwy tymczasowej (Staging). Naszą warstwą lądowania jest tabela `stg.iowa_liquor_sales_raw`. 
Przyjrzyjmy się celowo "zepsutym" rekordom, z którymi nasz system musiał się uporać:


In [ ]:
sql_dirty = """
SELECT TOP 5 
    invoice_and_item_number,
    date,
    store_location,      -- Problem: Format przestrzenny zapisany jako tekst POINT(X Y)
    category_name,       -- Problem: Brak przypisanej nazwy kategorii
    state_bottle_cost,   -- Problem: Symbol waluty $ i przecinki, uniemożliwiające agregację (SUM)
    sale_dollars         -- Problem: Zapisane jako łańcuch znaków
FROM stg.iowa_liquor_sales_raw
WHERE category_name IS NULL OR store_location LIKE 'POINT%';
"""
display(query_db(sql_dirty))


### Logika czyszczenia (Python / Pandas w procesie Loadera)
Zanim te dane trafią dalej, nasz system stosuje w Pythonie potężne techniki inżynierskie:
1. **Parsowanie wartości finansowych (`clean_money_or_number`)**: Użycie metod wektoryzowanych Pandas do usunięcia symbolu dolara i przecinków: `.str.replace('$', '').str.replace(',', '')`, rzutując ostatecznie łańcuch na natywny typ numeryczny.
2. **Ekstrakcja współrzędnych (`parse_location_value`)**: Zaawansowane wyrażenia regularne wyciągające czystą długość (`longitude`) i szerokość (`latitude`) geograficzną z ciągu typu POINT.
3. **Śledzenie zmian i deduplikacja (Hashowanie)**: Algorytm `SHA-256` spina wiele kolumn identyfikacyjnych, by wyliczyć unikalny `source_row_hash` dla każdego rekordu ze źródła. Odporne na luki i zmiany metadanych podejście.
4. **Masowe ładowanie danych**: Ekstremalnie szybki zapis poprzez natywny driver ODBC z flagą `cursor.fast_executemany = True` (ładujący dane paczkami, tzw. batch processing).


## 2. Warstwa Srebrna (Silver / Data Warehouse) - Model Gwiazdy i Operacje SQL
W kolejnym kroku tworzymy strukturę gotową pod zaawansowaną analizę (OLAP). Skrypty SQL w hurtowni przeprowadzają skomplikowaną inżynierię (Transform):
### Budowa Wymiarów (Deduplikacja)
Zarówno pliki CSV, jak i dane z tabeli tymczasowej pełne są duplikatów. By zbudować pojedynczy rekord dla każdego np. Sklepu (`dim_store`), użyliśmy funkcji okna (Window Functions). Zapytanie operacyjne wygląda następująco:
```sql
WITH ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY COALESCE(NULLIF(store_number, ''), 'UNKNOWN')
        ORDER BY date DESC, staging_key DESC
    ) AS rn
    FROM stg.iowa_liquor_sales_raw
)
INSERT INTO dw.dim_store ... SELECT ... FROM ranked WHERE rn = 1;
```
Dzięki `ROW_NUMBER() ... = 1`, w tabelach wymiarów (Dimension Tables) zjawia się unikalny rekord odpowiadający najnowszemu stanowi (najnowsza data) z systemu transakcyjnego.
### Obsługa braków (Zabezpieczenie The "Unknown" Member)
Model wymiarowy (Kimball) zakłada restrykcyjną zasade: **Tabela Faktów nigdy nie powinna mieć pustych (NULL) kluczy obcych**. By zadośćuczynić temu prawidłu, każde złączenie w warstwie ETL jest owrapowane klauzulą `COALESCE`: `COALESCE(NULLIF(raw.category, ''), 'UNKNOWN')`. System sztucznie zakłada rekord o nazwie "UNKNOWN" z kluczem równym "UNKNOWN", kierując wszystkie osierocone fakty do bezpiecznego punktu bez utraty wyliczeń finansowych.
Zobaczmy efekty:


In [ ]:
print("1. Wymiar Sklepu po ekstrakcji na czyste numeryczne współrzędne Latitude i Longitude:")
display(query_db("SELECT TOP 3 store_key, store_name, latitude, longitude FROM dw.dim_store WHERE latitude IS NOT NULL;"))
print("\n2. Defensywna technika wymiarowa: Rekord UNKNOWN dbający o relacyjną integralność faktów:")
display(query_db("SELECT * FROM dw.dim_category WHERE category_number = 'UNKNOWN';"))


### Tabela Faktów (Zdenormalizowana Moc Obliczeniowa)
Złączenie wyczyszczonych tabel do tabeli `dw.fact_sales`. Zamiast pisać złożoną analitykę w zewnętrznych aplikacjach (np. BI), wstrzykujemy logikę bezpośrednio w hurtownię. Na etapie budowy tabeli wyliczamy zysk z każdej pojedynczej butelki tworząc metrykę `margin_amount` jako `(state_bottle_retail - state_bottle_cost) * bottles_sold`.


In [ ]:
print("Fakty i pre-kalkulowana stopa zwrotu:")
display(query_db("SELECT TOP 5 invoice_number, store_key, category_key, sale_dollars, state_bottle_cost, margin_amount FROM dw.fact_sales;"))


## 3. Warstwa Złota (Gold / Semantic Layer) - 16 Biznesowych Widoków (Datamarts)
Gdy nasza hurtownia jest wymodelowana i zintegrowana, moglibyśmy oddać analitykom bazę na tacy. Zamiast tego wdrażamy **Warstwę Semantyczną**, co daje potrójną korzyść:
1. **Centralizacja Prawdy (Single Source of Truth)**: Metryki i zaawansowana kalkulacja rynkowa jest uwięziona i policzona raz na serwerze bazodanowym. Oznacza to, że każde zewnętrzne urządzenie zada to samo pytanie, a nie popełni błędu przeliczając coś "po swojemu".
2. **Ukrycie architektonicznej sieci złączeń (JOIN-ów)**: Analityk z użyciem narzędzi No-Code (Tableau) rzuca na kanwę jedno źródło. Złożoność jest ukryta za interfejsem widoku `vw_*`.
3. **Kategoryzacja Danych pod Tematy (Data Marts)**: Stworzyliśmy i zarządzamy 16 różnymi widokami. Składają się one w 4 odrębne domeny analityczne.
Poniżej przeanalizujemy każdy widok z osobna.


### Grupa 1: Ogólny Status i Wskaźniki Biznesowe (High-Level/ETL)
Zbiory stworzone pod użytek Zarządu (CEO, CFO) oraz zespołu technicznego Data Engineering (do audytu bazy).
#### 1. `sem.vw_kpi_summary`
Silnie zagregowany widok biznesowy obliczający uniwersalne wskaźniki (KPI - Key Performance Indicators) całego podmiotu. Liczy on na poziomie silnika bazodanowego średnie przychody, procentowe marże operacyjne oraz wolumen litrażu.


In [ ]:
display(query_db("SELECT * FROM sem.vw_kpi_summary;"))


#### 2. `sem.vw_etl_status`
Diagnostyka potoku ETL pokazująca aktualny stan zapełnienia (liczba rekordów) tabeli faktów i wszystkich wymiarów. Konieczna przy porannych przeglądach technicznych by natychmiast wychwycić błędy w zasileniach DWH.


In [ ]:
display(query_db("SELECT * FROM sem.vw_etl_status;"))


### Grupa 2: Domenowy Wymiar Czasu (Time-Series Analysis)
Widoki grupujące fakty ściśle po osiach temporalnych. Wymiar daty `dim_date` to silne narzędzie Kimballa pozwalające uniknąć stosowania dziesiątek funkcji operujących na dacie w bazowym SQL-u.
#### 3. `sem.vw_sales_overview`
Gigantyczny płaski widok ogólny. Wycina i zastępuje w pełni techniczne klucze sztuczne (Surrogate Keys, te kończące się na `_key`) prawdziwymi wartościami zrozumiałymi dla człowieka, dając możliwość analizowania wszystkich trendów czasowych na poziomie wiersza, z uwzględnieniem geolokalizacji.


In [ ]:
display(query_db("SELECT TOP 5 date, invoice_number, store_name, category_name, sale_dollars FROM sem.vw_sales_overview;"))


#### 4. `sem.vw_sales_by_day_type`
Wybitne zastosowanie inteligentnych struktur w Data Warehouse. Narzędzie grupuje zysk w oparciu o przypisaną do wymiaru czasu flagę `is_weekend`, dając błyskawiczną odpowiedź (zsumowane wartości finansowe) dla analizy różnicy zapotrzebowania w wolne dni rynkowe.


In [ ]:
display(query_db("SELECT * FROM sem.vw_sales_by_day_type;"))


#### 5. `sem.vw_sales_by_month`
Najpopularniejsza, uniwersalna struktura w analityce OLAP: Agregacja w hierarchii Rok -> Miesiąc. Widok gotowy, by rzucić go na wykres liniowy w BI (np. w używanym przez nas frameworku Streamlit).


In [ ]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_month ORDER BY year, month;"))


#### 6. `sem.vw_avg_sales_per_store_by_month_region`
Krzyżuje strukturę czasu ze strukturą przestrzenną. Odpowiada na rozbudowane, trudne zadania zarządcze: "Ile średnio (Average) na punkt przynosi miesiąc x w Hrabstwie y?".


In [ ]:
display(query_db("SELECT TOP 5 * FROM sem.vw_avg_sales_per_store_by_month_region;"))


### Grupa 3: Domenowy Wymiar Asortymentu i Opłacalności (Products & Margins)
Wyliczenia zysku (rentowności) w rozbiciu na dostawców, kategorie produktów i rozmiary dystrybuowanych opakowań.
#### 7. `sem.vw_sales_by_category`
Złoty klasyk agregacji. Warto tu zaznaczyć inżynieryjne zastosowanie `CROSS JOIN` ze zliczoną wcześniej całkowitą wartością przychodów (totals), w celu pre-kalkulacji parametru `sales_share_percent`! System analityczny nie musi dzielić na "swoim domowym kalkulatorze" części przez sumę, dostaje obrobiony wynik bezpośrednio z warstwy semantycznej.


In [ ]:
display(query_db("SELECT TOP 5 category_name, total_sales, sales_share_percent, avg_margin_per_bottle FROM sem.vw_sales_by_category ORDER BY total_sales DESC;"))


#### 8. `sem.vw_category_sales_over_time`
Widok wykorzystywany głównie w uczeniu maszynowym do wykrywania trendów rynkowych, spadków i analizy predykcyjnej, wiążący wskaźnik obrotu na produkcie z siatką czasową rynkową.


In [ ]:
display(query_db("SELECT TOP 5 * FROM sem.vw_category_sales_over_time;"))


#### 9. `sem.vw_top_products`
Tabela dedykowana odpytywaniu na bardzo głębokim ziarnie produktowym. Klasyczny ranking "Najlepiej sprzedający się konkretny produkt/kod" wsparty wolumenem litrażu (szczególnie krytycznym z perspektywy optymalizacji logistyki wysyłek).


In [ ]:
display(query_db("SELECT TOP 5 item_description, total_bottles_sold, total_sales FROM sem.vw_top_products ORDER BY total_sales DESC;"))


#### 10. `sem.vw_margin_analysis`
Filar opłacalności naszego biznesu. Uśredniony koszt zakupu hurtowego zderzony ze stawką dystrybucyjną rynkową by ujawnić marżę na pojedynczą jednostkę towarową (butelkę). Skalowane do totalnych sum dla analiz rentowności dostaw.


In [ ]:
display(query_db("SELECT TOP 5 category_name, vendor_name, avg_unit_margin, total_margin FROM sem.vw_margin_analysis ORDER BY total_margin DESC;"))


#### 11. `sem.vw_sales_by_packaging`
Agreguje rynkowe przychody do grup wolumenowych pojemników, by wyliczyć optymalne wielkości zamawianych zapasów od dostawców (koreluje z naszymi wewnętrznymi klasyfikatorami małpka/standardówka, zaszytymi w wymiarze `dim_packaging`).


In [ ]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_packaging ORDER BY total_sales DESC;"))


#### 12. `sem.vw_sales_by_vendor`
Rozliczenie kwot, prowizji i wykaz obrotów handlowych spójnych z gigantycznymi dostawcami alkoholu dostarczającymi zamówienia do Iowa (np. Diageo, Sazerac).


In [ ]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_vendor ORDER BY total_sales DESC;"))


### Grupa 4: Domenowy Wymiar Przestrzenny (Geospatial & Stores)
Mapowanie danych sprzedażowych na konkretne koordynaty, sklepy i hrabstwa pozwalające na tworzenie tzw. heat-map.
#### 13. `sem.vw_sales_by_geography`
Górny Roll-up terytorialny. Pozwala na budowanie kartogramów bazujących na miastach oraz przynależnych do nich Hrabstwach, umożliwiając wyodrębnienie obszarów o silnej opłacalności wewnątrzkrajowej.


In [ ]:
display(query_db("SELECT TOP 5 county, city, total_sales, total_bottles_sold FROM sem.vw_sales_by_geography ORDER BY total_sales DESC;"))


#### 14. `sem.vw_sales_map_points`
Prawdziwe dane wejściowe dla bibliotek GIS-owych. Widok dba o to by nie doszło do rzucenia wyjątkiem na warstwie wizualizacyjnej ze względu na złe dane - wymusza silnym filtrem z SQL `WHERE latitude IS NOT NULL AND longitude IS NOT NULL` tylko bezbłędne ukształtowanie rekordów.


In [ ]:
display(query_db("SELECT TOP 5 store_name, city, latitude, longitude, total_sales FROM sem.vw_sales_map_points ORDER BY total_sales DESC;"))


#### 15. `sem.vw_sales_by_store`
Zestawienie indywidualnej efektywności finansowej konkretnych punktów handlu. Bazy informacyjne pod wypłaty i premie dla operatorów sklepów.


In [ ]:
display(query_db("SELECT TOP 5 store_name, total_sales, total_margin, invoice_count FROM sem.vw_sales_by_store ORDER BY total_sales DESC;"))


#### 16. `sem.vw_volume_vs_revenue`
Zaawansowana analityka wielowymiarowa. Poszukuje korelacji gęstości dostaw transportu mierzonych w ciężarach / uciągach (Gallons, Liters) w konfrontacji do zwracanych marż finansowych, aby wesprzeć algorytmy decyzyjne zespołu Supply Chain / łańcucha dostaw.


In [ ]:
display(query_db("SELECT TOP 5 city, total_liters, total_revenue, avg_revenue_per_liter FROM sem.vw_volume_vs_revenue ORDER BY total_revenue DESC;"))
